In [1]:
import re
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, Trainer, TrainingArguments, BertForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df=pd.read_json('../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json')
df.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"


In [3]:
df.rename(
    columns={
        '도메인':' 도메인'
    }, inplace=True
)

df.head(1)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"


In [4]:
#데이터프레임 로드하고 컬럼의 이름들을 확인!
df.columns.str.strip()

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='str')

In [5]:
df.columns.map(lambda x : x.strip())

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='str')

In [6]:
df.columns=[x.strip() for x in df.columns]

In [7]:
#필요한 컬럼을 제외하고 나머지는 제외
df2=df[['고객질문(요청)', '상담사답변']]

In [8]:
df2.rename(
    columns={
        '고객질문(요청)' : '고객질문'
    },inplace=True
)

In [9]:
#텍스트 정규화
def normalize(text):
    text=re.sub(r'[^가-핳0-9a-zA-Z\s\.]',' ',str(text))
    text=re.sub(r'\s+',' ',text).strip()

    return text

df2=df2.map(normalize)

In [10]:
flag1=(df2['고객질문']!='') & (df2['상담사답변'].shift(-1)!='') & (df['문장번호']==1)
df3=df2.loc[flag1,]

In [11]:
#flag1을 한칸씩 앞으로 내리면 상답사의 답변
df3['상담사답변']=df2.loc[flag1.shift(1).fillna(False),'상담사답변'].tolist()
df3.head()

,고객질문,상담사답변
0,지방세를 내려면 어떻게 야됩니까,이용하시는 은 의 사이트에서 지방세 납부가 가능합니다.
20,지방세는 조 할 수 있습니까,간단한 본인 인 안내 드리겠습니다.
40,서울시주최 페스티벌 예매 놨는데 예정대로 진 됩니까,재로썬 진 될 예정입니다.
60,청년저축계좌 지금 신청할 수 있습니까,죄송하지만 이미 신청기간이 지났습니다.
200,보건증 무인발급기로 출력할 수 있습니까,보건증은 모든 보건소에서 지원하는게 아니라서 검사받은 보건소 인이 필요합니다.


In [12]:
df3.info()

<class 'pandas.DataFrame'>
Index: 1087 entries, 0 to 50316
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   고객질문    1087 non-null   str  
 1   상담사답변   1087 non-null   str  
dtypes: str(2)
memory usage: 146.1 KB


In [13]:
#기존의 데이터의 질문과 답변은 정상적인 답변 labels를 1로 채워준다.
df3['labels']=1

In [14]:
#기존의 질문을 유지하면서 답변은 바꿕서 labels가 0인 구간을 생성
answer_list=[
    ['A','a'],
    ['B','b'],
    ['C','c'],
    ['D','d']
]
neg_list=[]
for q, a in answer_list:
    #q : 질문
    #a : 답변
    cand=[]
    for q2,a2 in answer_list:
        #a2 : 답변들의 목록
        if a != a2:
            cand.append(a2)
    #cand 틀린 답변의 목록에서 무작위로 하나를 선택
    neg_a=np.random.choice(cand)
    neg_list.append([q, neg_a])

neg_list

[['A', np.str_('b')],
 ['B', np.str_('d')],
 ['C', np.str_('d')],
 ['D', np.str_('c')]]

In [15]:
answer_list2=df3[['고객질문','상담사답변']].values.tolist()

In [ ]:
neg_list2 = []
for q, a in answer_list2:
    cand = [ a2 for q2, a2 in answer_list2 if a != a2 ]

    neg_a = np.random.choice(cand)
    neg_list2.append( [q, neg_a] )
neg_list2

In [17]:
neg_df = pd.DataFrame(neg_list2, columns = ['고객질문', '상담사답변'])
neg_df['labels'] = 0
neg_df.head()

,고객질문,상담사답변,labels
0,지방세를 내려면 어떻게 야됩니까,네 무인민원발급기로 가족관계 증명서 발급이 가능하십니다.,0
1,지방세는 조 할 수 있습니까,결과는 9월 20일로 예정되어있습니다.,0
2,서울시주최 페스티벌 예매 놨는데 예정대로 진 됩니까,네 말씀하세요.,0
3,청년저축계좌 지금 신청할 수 있습니까,네 상담 드리겠습니다,0
4,보건증 무인발급기로 출력할 수 있습니까,주민센터에서 하실 수 있습니다.,0


In [18]:
# df3와 neg_df을 단순 행 결합 
dataset_df = pd.concat(
    [df3.head(500), neg_df.head(500)], axis = 0, ignore_index=True
)
dataset_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   고객질문    1000 non-null   str  
 1   상담사답변   1000 non-null   str  
 2   labels  1000 non-null   int64
dtypes: int64(1), str(2)
memory usage: 131.7 KB


In [19]:
dataset_df['labels'].value_counts()

labels
1    500
0    500
Name: count, dtype: int64

In [20]:
train_df,test_df = train_test_split(
    dataset_df, test_size=0.5, random_state=42, stratify=dataset_df['labels']
)

In [21]:
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))
#DatasetDict
ds=DatasetDict(
    {
        'train' : train_ds,
        'validation' : test_ds
    }
)
ds

DatasetDict({
    train: Dataset({
        features: ['고객질문', '상담사답변', 'labels'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['고객질문', '상담사답변', 'labels'],
        num_rows: 500
    })
})

In [24]:
MODEL_NAME='beomi/kcbert-base'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_first=True)
max_len=128

def token_fn(batch):
    #batch -> dict('고객질문','상담사답변)
    tok=tokenizer(
        batch['고객질문'],
        batch['상담사답변'],
        trucation=True,
        max_length=max_len
    )

    #토큰화된 데이터에서 token_type_ids는 kobert 모델에서는 사용하지 않는다.
    #해당 키를 제거
    tok.pop('token_type_ids', None)
    return tok

#remove_columns 매개변수 : 토큰화를 하고 제외시킬 컬럼을 지정
tok_ds=ds.map(
    token_fn,
    batched=True,
    # remove_columns=['고객질문', '상담사답변']
    remove_columns=[col for col in dataset_df.columns if col not in ['labels']],
    #캐시 사용 안함
    load_from_cache_file=True
)

Map: 100%|██████████| 500/500 [00:00<00:00, 2819.24 examples/s]


In [ ]:
tok_ds['train'][0]['input_ids']

In [35]:
#배치마다 동적으로 padding 토큰을 추가
collator=DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [36]:
#학습 모델 정의
#BertModel -> dropout -> Linear
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1285.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [37]:
#평가 함수
def metrics(eval_pred):
    logits, y=eval_pred
    pred=np.argmax(logits, axis=-1)
    return {
        'accuracy_score' : accuracy_score(pred,y),
        'f1_score' : f1_score(pred,y)
    }

In [40]:
args=TrainingArguments(
    output_dir='/model2',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=0.1,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_score',
    greater_is_better=True
)

In [41]:
trainer = Trainer(
    model = model, 
    args = args, 
    train_dataset= tok_ds['train'], 
    eval_dataset= tok_ds['validation'], 
    processing_class= tokenizer, 
    data_collator= collator, 
    compute_metrics= metrics
)
trainer.train()

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,No log,0.685784,0.506000,0.518519
2,No log,0.819342,0.526000,0.551985
3,No log,1.107249,0.554000,0.608084


Writing model shards: 100%|██████████| 1/1 [00:11<00:00, 11.29s/it]
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:27<00:00, 27.26s/it]
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.19s/it]


TrainOutput(global_step=189, training_loss=0.5159779705067791, metrics={'train_runtime': 1911.1767, 'train_samples_per_second': 0.785, 'train_steps_per_second': 0.099, 'total_flos': 27974049628080.0, 'train_loss': 0.5159779705067791, 'epoch': 3.0})